# FARZ | فَرز — PyTorch Training (Google Colab)

This notebook prepares the approved five-class TrashNet subset, creates a leakage-aware split, trains **MobileNetV3-Small** with PyTorch, and evaluates once on the held-out test set.

**Classes:** Glass · Paper · Cardboard · Plastic · Metal  
**Trash is excluded.** The six photos involved in three exact cross-label duplicate pairs are held out pending label review. No original photos are deleted or modified.

**Recommended:** Runtime → Change runtime type → GPU (T4 if available).

Dataset: Gary Thung and Mindy Yang's [original TrashNet project](https://github.com/garythung/trashnet) and the author's [Hugging Face dataset](https://huggingface.co/datasets/garythung/trashnet). The author asks users to cite the original project. The notebook downloads the resized archive inside Colab; no dataset transfer through your laptop is needed.

In [ ]:
# 1) Upload the small FARZ ML source bundle.
# Download FARZ-ML-Training-Source.zip from this conversation, then select it here.
from google.colab import files
from pathlib import Path
import zipfile, shutil, os

uploaded = files.upload()
source_zip = next((name for name in uploaded if name.lower().endswith(".zip")), None)
if source_zip is None:
    raise FileNotFoundError("Upload FARZ-ML-Training-Source.zip")

PROJECT = Path("/content/FARZ-AI")
if PROJECT.exists():
    shutil.rmtree(PROJECT)
with zipfile.ZipFile(source_zip) as archive:
    archive.extractall("/content")

if not (PROJECT / "ml/src/training/train.py").is_file():
    raise FileNotFoundError("Source ZIP must contain FARZ-AI/ml/src/training/train.py")
os.chdir(PROJECT)
print("FARZ source ready:", PROJECT)

## 2) Prepare the Colab runtime

PyTorch and Torchvision are already installed in standard Colab GPU runtimes. We install the remaining libraries used by the pipeline.

In [ ]:
%pip -q install -r /content/FARZ-AI/ml/requirements.txt

import torch, torchvision, sklearn
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Select a GPU runtime before starting model training.")

## 3) Download TrashNet directly to Colab

This downloads the author's 42.8 MB resized archive to Colab's temporary disk and extracts it. It does not download the images to your laptop first.

In [ ]:
from pathlib import Path
import subprocess

DATA_URL = "https://huggingface.co/datasets/garythung/trashnet/resolve/main/dataset-resized.zip"
ZIP_PATH = Path("/content/dataset-resized.zip")
RAW_DIR = PROJECT / "data/raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

if not ZIP_PATH.exists():
    subprocess.run(["wget", "-q", "--show-progress", "-O", str(ZIP_PATH), DATA_URL], check=True)

print(f"Archive size: {ZIP_PATH.stat().st_size / (1024*1024):.1f} MiB")
subprocess.run(["unzip", "-q", "-o", str(ZIP_PATH), "-d", str(RAW_DIR)], check=True)
IMAGES_DIR = RAW_DIR / "dataset-resized"
assert IMAGES_DIR.is_dir(), f"Missing extracted folder: {IMAGES_DIR}"
print("Available image folders:", sorted(p.name for p in IMAGES_DIR.iterdir() if p.is_dir()))

## 4) Audit labels and build a new split

The audit selects only the five approved class folders, checks file integrity, and computes SHA-256 hashes. We then hold out all six members of the three exact cross-label duplicate pairs. The final split keeps the remaining perceptual-hash candidate pairs together so they cannot cross between train, validation, and test.

In [ ]:
import subprocess, sys

def run(command):
    print(">>", " ".join(map(str, command)))
    subprocess.run(list(map(str, command)), cwd=PROJECT, check=True)

run([sys.executable, "ml/src/data/audit_dataset.py",
     "--images-dir", "data/raw/dataset-resized", "--root", "."])
run([sys.executable, "ml/src/data/build_training_candidates.py", "--root", "."])
run([sys.executable, "ml/src/data/create_data_split.py", "--root", ".", "--seed", "42"])

In [ ]:
# 5) Verify the split before training.
import json, pandas as pd

summary = json.loads((PROJECT / "reports/data_audit/split_counts.json").read_text())
print(json.dumps(summary, indent=2))
split_df = pd.read_csv(PROJECT / "data/manifests/data_split.csv")
display(split_df.groupby(["split", "class_name"]).size().unstack(fill_value=0))
assert len(split_df) == 2384
assert split_df.groupby("group_id")["split"].nunique().max() == 1, "Data leakage detected"
assert set(split_df["class_name"].unique()) == {"Glass", "Paper", "Cardboard", "Plastic", "Metal"}

## 6) Train MobileNetV3-Small

The model starts from ImageNet pretrained weights and is fine-tuned on FARZ's five classes. Validation macro-F1 selects the best checkpoint. The test split is not used for training or early stopping.

Default settings: at most 20 epochs, batch size 32, input size 224, learning rate 0.0003, early stopping patience 5. Mixed precision is enabled automatically on CUDA.

In [ ]:
run([
    sys.executable, "ml/src/training/train.py", "--root", ".",
    "--epochs", "20", "--batch-size", "32", "--image-size", "224",
    "--learning-rate", "0.0003", "--patience", "5", "--num-workers", "2"
])

## 7) Evaluate once on the untouched test split

Run this after training. The output contains accuracy, macro precision/recall/F1, per-class metrics, a confusion matrix, and individual test predictions. Softmax scores are not calibrated probabilities.

In [ ]:
run([sys.executable, "ml/src/evaluation/evaluate.py", "--root", ".",
     "--batch-size", "32", "--num-workers", "2"])

from IPython.display import display, Image
metrics_path = PROJECT / "ml/outputs/mobilenet_v3_small/test_evaluation/test_metrics.json"
print(metrics_path.read_text())
cm_path = PROJECT / "ml/outputs/mobilenet_v3_small/test_evaluation/confusion_matrix.png"
display(Image(filename=str(cm_path)))

## 8) Save the checkpoint and reports to Google Drive

Mount Drive so the checkpoint and experiment outputs remain available after the Colab runtime is reset.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DEST = Path("/content/drive/MyDrive/FARZ-Results/MobileNetV3-Small")
DEST.mkdir(parents=True, exist_ok=True)
SOURCE_RESULTS = PROJECT / "ml/outputs/mobilenet_v3_small"
shutil.copytree(SOURCE_RESULTS, DEST, dirs_exist_ok=True)
print("Saved FARZ results to:", DEST)
print("Files:", sorted(p.name for p in DEST.iterdir()))

## 9) Before reporting results

- Report the final test metrics only after the held-out test evaluation.
- Include macro-F1, per-class precision/recall/F1, and the confusion matrix, not accuracy alone.
- Inspect errors between Paper and Cardboard, and between Glass and Plastic.
- Record GPU type, hyperparameters, best validation epoch, checkpoint size, and training time.
- TrashNet contains single waste items against mostly simple backgrounds. Do not claim the same performance for cluttered real-world scenes.

**Next step after training:** export the checkpoint for on-device inference and validate PyTorch/ExecuTorch output agreement on an Android phone before connecting the Flutter camera screen.